# 일정 관리 챗봇 만들기

이 노트북을 순서대로 따라가면 **자연어로 일정을 추가·조회·삭제할 수 있는 AI 챗봇**이 완성됩니다!

```
나: 내일 오후 3시에 팀 미팅 추가해줘
AI: 팀 미팅 일정을 추가했어요! (2026-05-14 15:00)

나: 이번 주 일정 전부 보여줘
AI: 이번 주 일정 목록입니다...
```

**완성까지의 순서**
1. 환경 설정
2. 데이터 저장 헬퍼 함수 만들기
3. 에이전트 도구(Tool) 4개 만들기
4. 도구를 LLM 없이 직접 테스트
5. 시스템 프롬프트 작성
6. 에이전트 조립
7. 에이전트 테스트
8. 미들웨어 추가 (로깅 + 토큰 추적)

![프로젝트구조](./schedule_chatbot.png)

## Step 1. 환경 설정

필요한 라이브러리를 불러오고 LLM 모델을 정의합니다.

In [ ]:
import json
import os
import uuid
from datetime import date

from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import wrap_tool_call, after_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

load_dotenv("../.env")

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

model = ChatOpenAI(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticekt': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
)

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## Step 2. 데이터 저장 헬퍼 함수

일정을 `data/schedules.json` 파일에 저장하고 불러오는 내부 함수 2개를 정의합니다.

이 함수들은 @tool로 직접 노출하지 않고, 아래 도구들 내부에서만 사용합니다.

In [2]:
DATA_PATH = "./data/schedules.json"

def _load_schedules():
    """JSON 파일에서 일정 목록을 불러옵니다."""
    if not os.path.exists(DATA_PATH):
        return []
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def _save_schedules(schedules):
    """일정 목록을 JSON 파일에 저장합니다."""
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    with open(DATA_PATH, "w", encoding="utf-8") as f:
        json.dump(schedules, f, ensure_ascii=False, indent=2)

## Step 3. 도구(Tool) 만들기

`@tool` 데코레이터를 사용해 LLM이 사용할 수 있는 도구를 정의합니다.

도구를 만들 때 주의사항:
- 인자 타입과 반환 타입을 반드시 명시
- docstring을 꼭 작성 (LLM이 이 설명을 읽고 언제 어떻게 도구를 쓸지 판단합니다)

만들 도구 목록:
1. `add_schedule` - 일정 추가
2. `get_schedules` - 특정 날짜 일정 조회
3. `list_all_schedules` - 전체 일정 조회
4. `delete_schedule` - 일정 삭제

### 도구 1: add_schedule

날짜, 시간, 제목, 설명을 받아 새로운 일정을 추가합니다.

`uuid.uuid4()`로 고유 ID를 생성해서 저장하면 나중에 삭제할 때 정확하게 찾을 수 있습니다.

In [3]:
@tool
def add_schedule(date: str, time: str, title: str, description: str = "") -> str:
    """
    새로운 일정을 추가합니다.
    - date: 날짜 (예: 2024-01-15)
    - time: 시간 (예: 14:00)
    - title: 일정 제목
    - description: 상세 설명 (선택사항)
    """
    schedules = _load_schedules()

    new_schedule = {
        "id": str(uuid.uuid4())[:8],
        "date": date,
        "time": time,
        "title": title,
        "description": description,
    }

    schedules.append(new_schedule)
    _save_schedules(schedules)

    return f"일정이 추가되었습니다! (ID: {new_schedule['id']}) {date} {time} - {title}"

In [4]:
@tool
def get_schedules(date: str) -> str:
    """
    특정 날짜의 일정을 조회합니다.
    - date: 조회할 날짜 (예: 2024-01-15)
    """
    schedules = _load_schedules()

    day_schedules = [s for s in schedules if s["date"] == date]

    if not day_schedules:
        return f"{date}에 등록된 일정이 없습니다."

    day_schedules.sort(key=lambda x: x["time"])

    result = f"[{date} 일정 목록]\n"
    for s in day_schedules:
        result += f"- {s['time']} {s['title']}"
        if s["description"]:
            result += f" ({s['description']})"
        result += f" [ID: {s['id']}]\n"

    return result

In [5]:
@tool
def list_all_schedules() -> str:
    """저장된 모든 일정을 날짜 순서대로 조회합니다."""
    schedules = _load_schedules()

    if not schedules:
        return "저장된 일정이 없습니다."

    schedules.sort(key=lambda x: (x["date"], x["time"]))

    result = "[전체 일정]\n"
    current_date = ""
    for s in schedules:
        if s["date"] != current_date:
            current_date = s["date"]
            result += f"\n📅 {current_date}\n"
        result += f"  - {s['time']} {s['title']}"
        if s["description"]:
            result += f" ({s['description']})"
        result += f" [ID: {s['id']}]\n"

    return result

In [6]:
@tool
def delete_schedule(schedule_id: str) -> str:
    """
    일정을 삭제합니다.
    - schedule_id: 삭제할 일정의 ID (일정 조회 시 확인 가능)
    """
    schedules = _load_schedules()

    target = next((s for s in schedules if s["id"] == schedule_id), None)

    if not target:
        return f"ID '{schedule_id}'에 해당하는 일정을 찾을 수 없습니다. 먼저 일정을 조회해서 ID를 확인해주세요."

    schedules.remove(target)
    _save_schedules(schedules)

    return f"'{target['title']}' 일정이 삭제되었습니다. ({target['date']} {target['time']})"

## Step 4. 도구 직접 테스트 (LLM 없이)

에이전트를 만들기 전에 도구들이 제대로 동작하는지 **LLM 없이 직접 호출해서** 검증합니다.

이 단계에서 버그를 잡아두면 나중에 에이전트 수준에서 디버깅하는 것보다 훨씬 쉽습니다.

In [7]:
# @tool 데코레이터가 붙은 함수는 .invoke()로 직접 호출할 수 있습니다.

# 일정 추가 테스트
print(add_schedule.invoke({"date": "2026-05-20", "time": "10:00", "title": "팀 미팅"}))
print(add_schedule.invoke({"date": "2026-05-20", "time": "14:00", "title": "점심 약속", "description": "삼겹살 식당"}))
print(add_schedule.invoke({"date": "2026-05-21", "time": "09:00", "title": "조기 축구"}))

일정이 추가되었습니다! (ID: 880f940c) 2026-05-20 10:00 - 팀 미팅
일정이 추가되었습니다! (ID: 8354a099) 2026-05-20 14:00 - 점심 약속
일정이 추가되었습니다! (ID: 0de7f421) 2026-05-21 09:00 - 조기 축구


In [8]:
# 특정 날짜 조회 테스트
print(get_schedules.invoke({"date": "2026-05-20"}))

[2026-05-20 일정 목록]
- 10:00 팀 미팅 [ID: 880f940c]
- 14:00 점심 약속 (삼겹살 식당) [ID: 8354a099]



In [9]:
# 전체 조회 테스트
print(list_all_schedules.invoke({}))

[전체 일정]

📅 2026-05-14
  - 15:00 팀 미팅 [ID: a7582808]

📅 2026-05-20
  - 10:00 팀 미팅 [ID: 880f940c]
  - 14:00 점심 약속 (삼겹살 식당) [ID: 8354a099]

📅 2026-05-21
  - 09:00 조기 축구 [ID: 0de7f421]



In [10]:
# 삭제 테스트 (위에서 추가한 일정의 ID로 시도해 보세요)
print(delete_schedule.invoke({"schedule_id": "존재하지않는ID"}))

ID '존재하지않는ID'에 해당하는 일정을 찾을 수 없습니다. 먼저 일정을 조회해서 ID를 확인해주세요.


## Step 5. 시스템 프롬프트 작성

시스템 프롬프트는 에이전트의 행동 지침입니다. 다음 4가지를 포함하면 좋습니다.

1. **역할 정의** - 이 AI가 무엇을 하는 어시스턴트인지
2. **도구 사용 규칙** - 어떤 상황에 어떤 도구를 써야 하는지
3. **데이터 형식** - 날짜/시간 형식 등 일관성을 위한 규칙
4. **답변 규칙** - 어떻게 답변해야 하는지

또한 에이전트가 '오늘', '내일', '이번 주' 같은 상대적인 날짜를 계산하려면 오늘 날짜를 프롬프트에 포함해야 합니다.

In [11]:
today_str = date.today().strftime("%Y-%m-%d")

system_prompt = f"""# 일정 관리 AI 어시스턴트

## 역할
당신은 사용자의 일정을 관리하는 AI 어시스턴트입니다.
일정 추가, 조회, 삭제를 도와드리며, 항상 친절하고 간결하게 답변하세요.

## 오늘 날짜
{today_str}

## 도구 사용 규칙
- 일정 추가 요청 → add_schedule 사용
- 특정 날짜 조회 요청 → get_schedules 사용
- 전체 일정 조회 요청 → list_all_schedules 사용
- 일정 삭제 요청 → ID가 있으면 바로 delete_schedule, 없으면 먼저 조회해서 ID 확인 후 삭제

## 날짜/시간 형식
- 날짜는 반드시 YYYY-MM-DD 형식으로 저장 (예: 2024-01-15)
- 시간은 반드시 HH:MM 형식으로 저장 (예: 14:00)
- '내일', '다음 주 월요일' 등 상대적 날짜는 오늘 날짜를 기준으로 계산

## 답변 규칙
- 도구 실행 결과를 바탕으로 명확하게 답변하세요.
- 일정 추가/삭제 완료 후에는 결과를 간단히 요약해서 알려주세요.
- 사용자가 날짜/시간을 명확히 말하지 않았다면 먼저 확인하세요.
"""

print(system_prompt)

# 일정 관리 AI 어시스턴트

## 역할
당신은 사용자의 일정을 관리하는 AI 어시스턴트입니다.
일정 추가, 조회, 삭제를 도와드리며, 항상 친절하고 간결하게 답변하세요.

## 오늘 날짜
2026-05-13

## 도구 사용 규칙
- 일정 추가 요청 → add_schedule 사용
- 특정 날짜 조회 요청 → get_schedules 사용
- 전체 일정 조회 요청 → list_all_schedules 사용
- 일정 삭제 요청 → ID가 있으면 바로 delete_schedule, 없으면 먼저 조회해서 ID 확인 후 삭제

## 날짜/시간 형식
- 날짜는 반드시 YYYY-MM-DD 형식으로 저장 (예: 2024-01-15)
- 시간은 반드시 HH:MM 형식으로 저장 (예: 14:00)
- '내일', '다음 주 월요일' 등 상대적 날짜는 오늘 날짜를 기준으로 계산

## 답변 규칙
- 도구 실행 결과를 바탕으로 명확하게 답변하세요.
- 일정 추가/삭제 완료 후에는 결과를 간단히 요약해서 알려주세요.
- 사용자가 날짜/시간을 명확히 말하지 않았다면 먼저 확인하세요.



## Step 6. 에이전트 만들기

도구, 시스템 프롬프트, 메모리를 `create_agent()`로 묶어 에이전트를 만듭니다.

- `tools`: LLM이 사용할 수 있는 도구 목록
- `system_prompt`: 행동 지침
- `checkpointer`: 대화 메모리 (`InMemorySaver`는 실행 중에만 유지됨)

In [12]:
agent = create_agent(
    model,
    tools=[add_schedule, get_schedules, list_all_schedules, delete_schedule],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

## Step 7. 에이전트 테스트

`thread_id`를 동일하게 넘기면 이전 대화 내용을 기억합니다.

In [13]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "내일 오후 3시에 팀 미팅 추가해줘"}},
    {"configurable": {"thread_id": "1"}},
)
print(response["messages"][-1].content)

일정이 추가되었습니다.  
- 2026-05-14 15:00 팀 미팅  
- ID: 872df02a


In [14]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "내일 일정 알려줘"}},
    {"configurable": {"thread_id": "1"}},
)
print(response["messages"][-1].content)

내일(2026-05-14) 일정은 다음과 같습니다.

- 15:00 팀 미팅 [ID: 872df02a]


In [15]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "방금 추가한 팀 미팅 삭제해줘"}},
    {"configurable": {"thread_id": "1"}},
)
print(response["messages"][-1].content)

팀 미팅 일정을 삭제했습니다.  
- 2026-05-14 15:00 팀 미팅


## Step 8. 미들웨어 추가

미들웨어란 에이전트가 작업을 처리하기 전/후에 개입하는 기능입니다.

두 가지를 추가합니다:
1. **`@wrap_tool_call`** - 도구가 호출될 때마다 이름과 인수를 로그로 출력
2. **`@after_agent`** - 에이전트 응답이 완료될 때마다 토큰 사용량을 출력

미들웨어는 `create_agent(middleware=[...])`에 리스트로 넘겨서 적용합니다.

### 미들웨어 1: 도구 호출 로거

`@wrap_tool_call`을 함수 위에 붙이면 도구가 호출될 때마다 실행되는 미들웨어가 됩니다.

함수는 반드시 `request`와 `handler` 2개의 인자를 받아야 합니다.
- `request.tool_call["name"]` : 호출된 도구 이름
- `request.tool_call["args"]` : 도구에 전달된 인수
- `handler(request)` : 실제 도구를 실행하는 함수

In [16]:
@wrap_tool_call
def tool_logger(request, handler):
    name = request.tool_call["name"]
    args = request.tool_call["args"]
    print(f"\n  [TOOL] {name} | 인수: {args}")

    result = handler(request)
    print(f"  [TOOL] 결과: {result.content}")
    return result

### 미들웨어 2: 토큰 사용량 추적기

`@after_agent`를 붙이면 에이전트가 최종 응답을 생성할 때마다 실행됩니다.

`state["messages"][-1]`이 마지막 AI 응답 메시지이고,
`usage_metadata`에 토큰 사용량이 담겨 있습니다.

- `usage_metadata["input_tokens"]` : 입력 토큰 수
- `usage_metadata["output_tokens"]` : 출력 토큰 수

In [ ]:
_total = {"input": 0, "output": 0}

@after_agent
def token_tracker(state: AgentState, runtime: Runtime):
    last = state["messages"][-1]
    if not (hasattr(last, "usage_metadata") and last.usage_metadata):
        return None

    u = last.usage_metadata
    inp = u.get("input_tokens", 0)
    out = u.get("output_tokens", 0)
    _total["input"] += inp
    _total["output"] += out
    print(f"  [TOKEN] 누적: 입력 {_total['input']} / 출력 {_total['output']} / 합계 {_total['input'] + _total['output']}")
    return None

### 미들웨어를 포함한 에이전트 재생성

`middleware` 리스트에 넣는 순서대로 실행됩니다.

In [18]:
agent = create_agent(
    model,
    tools=[add_schedule, get_schedules, list_all_schedules, delete_schedule],
    system_prompt=system_prompt,
    middleware=[tool_logger, token_tracker],
    checkpointer=InMemorySaver(),
)

In [19]:
# 미들웨어가 동작하는 것을 확인해 봅시다!
test_messages = [
    "모레 오전 10시에 치과 예약 추가해줘",
    "이번 주 전체 일정 보여줘",
]

for msg in test_messages:
    print(f"나: {msg}")
    response = agent.invoke(
        {"messages": {"role": "user", "content": msg}},
        {"configurable": {"thread_id": "test"}},
    )
    print(f"AI: {response['messages'][-1].content}\n")

print("=" * 50)
print(f"[최종 토큰 합계] 입력 {_total['input']} / 출력 {_total['output']} / 합계 {_total['input'] + _total['output']}")

나: 모레 오전 10시에 치과 예약 추가해줘

  [TOOL] add_schedule | 인수: {'date': '2026-05-15', 'time': '10:00', 'title': '치과 예약'}
  [TOOL] 결과: 일정이 추가되었습니다! (ID: b599e47c) 2026-05-15 10:00 - 치과 예약
  [TOKEN] 이번: 입력 633 / 출력 39 / 합계 672
  [TOKEN] 누적: 입력 633 / 출력 39 / 합계 672
AI: 추가했어요.

- 일정: 치과 예약
- 일시: 2026-05-15 10:00
- ID: b599e47c

나: 이번 주 전체 일정 보여줘
  [TOKEN] 이번: 입력 685 / 출력 79 / 합계 764
  [TOKEN] 누적: 입력 1318 / 출력 118 / 합계 1436
AI: 이번 주는 기준이 조금 애매할 수 있어요.  
원하시면 아래 중 하나로 바로 조회해드릴게요.

1. **이번 주 월~일 전체**
2. **오늘(2026-05-13)부터 7일간**
3. **특정 날짜 범위 지정**

원하시는 기준을 말씀해 주세요.

[최종 토큰 합계] 입력 1318 / 출력 118 / 합계 1436
